# Limpieza del dataset Healthcare Dataset

Trabajo de la sesión de limpieza de datos · Minería de Datos

Dataset: Healthcare Dataset de Kaggle (55.500 admisiones hospitalarias).

## Introducción

El conjunto tiene 15 columnas: datos del paciente, de la admisión y de la parte administrativa de la cuenta. A primera vista se ve limpio porque no trae valores nulos, pero al revisarlo con calma aparecen filas repetidas, nombres con mayúsculas revueltas y facturas con montos negativos. Acá está el recorrido completo, usando las clases que están repartidas en los módulos de esta misma carpeta; cada una se encarga de una sola parte del proceso.

In [1]:
import sys
from pathlib import Path

CARPETA_CODIGO = Path.cwd() / "Codigo Limpieza de Datos"
if str(CARPETA_CODIGO) not in sys.path:
    sys.path.append(str(CARPETA_CODIGO))

import pandas as pd

from repositorios.cargador import CargadorDataset
from utilidades.constantes import ARCHIVO_CRUDO, ARCHIVO_LIMPIO, COLUMNA_FLAG_FACTURA_NEGATIVA
from servicios.calidad import Diagnosticador, AnalizadorOutliers, Validador
from servicios.limpieza import MarcadorFacturasNegativas, NormalizadorTexto, TratadorDuplicados

df = CargadorDataset(ARCHIVO_CRUDO).cargar()


## Diagnóstico inicial

Reviso lo de siempre: cuántas filas hay, qué está repetido y qué se sale de rango.

In [2]:
Diagnosticador(["Age", "Billing Amount"]).diagnosticar(df, "Dataset crudo")


=== Dataset crudo ===
Filas: 55500
Duplicados: 534
Nulos: 0
            Age  Billing Amount
count  55500.00        55500.00
mean      51.54        25539.32
std       19.60        14211.45
min       13.00        -2008.49
25%       35.00        13241.22
50%       52.00        25538.07
75%       68.00        37820.51
max       89.00        52764.28


No hay nulos, la edad va de 13 a 89 años (valores normales) y hay 534 filas repetidas. Lo que salta a la vista es el mínimo de Billing Amount: -2008.49, una factura no puede ser negativa.

## Duplicados

Se quitan primero porque si no, los cálculos de después (medianas, proporciones) quedan contando doble.

In [3]:
df = TratadorDuplicados().quitar(df)
print("Filas:", len(df))


Duplicados eliminados: 534
Filas: 54966


## Texto inconsistente

En las columnas de texto libre los nombres están con mayúsculas y minúsculas mezcladas al azar, por ejemplo:

In [4]:
df[["Name", "Doctor"]].head(6)


            Name            Doctor
0  Bobby JacksOn     Matthew Smith
1   LesLie TErRy   Samantha Davies
2    DaNnY sMitH  Tiffany Mitchell
3   andrEw waTtS       Kevin Wells
4  adrIENNE bEll    Kathleen Hanna
5  EMILY JOHNSOn     Taylor Newton


Ese tipo de diferencia hace que un mismo nombre no coincida en búsquedas exactas, así que dejo todo en formato de título con strip() y title().

In [5]:
df = NormalizadorTexto(["Name", "Doctor", "Hospital"]).normalizar(df)
df[["Name", "Doctor"]].head(6)


            Name            Doctor
0  Bobby Jackson     Matthew Smith
1   Leslie Terry   Samantha Davies
2    Danny Smith  Tiffany Mitchell
3   Andrew Watts       Kevin Wells
4  Adrienne Bell    Kathleen Hanna
5  Emily Johnson     Taylor Newton


## Facturas negativas

Este es el problema más serio: hay montos negativos en Billing Amount, un valor imposible para una factura. En el crudo son 108, y al quitar los duplicados quedan 106. Para ver el alcance primero los cuento.

In [6]:
negativos = df[df["Billing Amount"] < 0]
print("Facturas negativas:", len(negativos))


Facturas negativas: 106


Antes de marcarlos reviso si el error depende de alguna variable (por ejemplo, de un tipo de admisión) o si simplemente está repartido al azar.

In [7]:
print("Proporcion por tipo de admision (facturas negativas):")
print((negativos["Admission Type"].value_counts(normalize=True) * 100).round(1))
print()
print("Proporcion por tipo de admision (dataset completo):")
print((df["Admission Type"].value_counts(normalize=True) * 100).round(1))


Proporcion por tipo de admision (facturas negativas):
Urgent       35.8
Elective     34.0
Emergency    30.2
Name: proportion, dtype: float64

Proporcion por tipo de admision (dataset completo):
Elective     33.6
Urgent       33.5
Emergency    32.9
Name: proportion, dtype: float64


Las proporciones son casi iguales, y pasa lo mismo revisando aseguradora y condición médica. Es decir, el error no depende de ninguna variable observada, parece un error de captura al azar (MCAR). Pero que sea aleatorio no da licencia para inventar el valor: esto es dinero, un dato sensible, y reemplazar el monto con una mediana fabricaría una factura que la empresa nunca registró y podría generar pérdidas. La decisión fue marcar las 106 facturas con una columna aparte y dejarlas fuera de todos los cálculos, sin tocar el monto original.

In [8]:
df = MarcadorFacturasNegativas(COLUMNA_FLAG_FACTURA_NEGATIVA).marcar(df)
print("Marcadas == negativas:", int(df[COLUMNA_FLAG_FACTURA_NEGATIVA].sum()) == int((df["Billing Amount"] < 0).sum()))


Facturas negativas marcadas: 106
El monto original se conserva, no se imputa ningún valor.
Marcadas == negativas: True


## Outliers (regla IQR)

Reviso las columnas numéricas con la regla de Tukey, que usa los cuartiles y el rango intercuartílico con un factor de 1.5. Las facturas negativas marcadas quedan excluidas de este análisis: su valor no es una medición válida de facturación.

In [9]:
AnalizadorOutliers(["Age", "Billing Amount"], COLUMNA_FLAG_FACTURA_NEGATIVA).revisarIQR(df)


Age: limites [-14.50, 117.50], fuera de rango: 0
Billing Amount: limites [-23521.23, 74668.04], fuera de rango: 0


No hubo valores fuera de rango y no hubo que recortar nada. Un detalle: sobre los valores válidos el límite inferior de Billing Amount da -23521.23 y el mínimo real es 9.24, así que ningún dato queda fuera. La regla IQR nunca iba a detectar las facturas negativas porque no son valores extremos de la distribución; ese problema solo lo atrapa la regla de negocio (una factura no puede ser negativa), y por eso se marcan aparte.

## Validación y exportación

Antes de guardar el archivo reviso que se cumplan las condiciones mínimas: sin duplicados, sin nulos, todas las facturas negativas marcadas y con el monto original intacto.

In [10]:
Validador(COLUMNA_FLAG_FACTURA_NEGATIVA).validar(df)
df.to_csv(ARCHIVO_LIMPIO, index=False, float_format="%.2f")
print("Guardado. Filas:", len(df))


Duplicados: 0
Nulos: 0
Facturas negativas: 106
Todas marcadas: True
Monto original conservado: True
Edades fuera de rango: 0
Guardado. Filas: 54966


## Conclusiones

El dataset no tenía nulos pero igual había que limpiarlo: lo importante no era contar NaN sino revisar la calidad completa. El problema real fueron los montos negativos, y la lección de fondo es que en datos sensibles (dinero) no se inventan valores: las facturas negativas se marcaron y se excluyeron de los cálculos, conservando intacto el monto original para auditoría. También quedó clara una cosa de la teoría: la estadística sola no alcanza, el IQR no detecta los negativos porque no son valores extremos para la distribución, ese problema solo lo ve la regla de negocio. Y el orden del proceso importa: si no se quitan primero los duplicados, los conteos y las medianas salen mal. Se terminó con 54966 filas, 0 duplicados, 0 nulos y las 106 facturas negativas marcadas.